In [0]:
%pip install --upgrade databricks-sdk pyyaml jsonschema
dbutils.library.restartPython()

# Init Ontology Project

Validates the Genie Ontology SDK inputs (`sdk_config.json` contract), checks Unity Catalog access, then scaffolds a run project (both bundles, run kit, docs folders, vendored library) with `genie_ontology_sdk.scaffold.scaffold_project`.

* List widgets take comma-separated values.
* Leave `target_dir` empty to run only the validation and access check.
* Existing files are never overwritten unless `force` is `true`; `.databricks/` (including `variable-overrides.json`) is never touched.

In [0]:
# One widget per sdk_config.json field (schemas/sdk_config.schema.json), plus scaffold options.
dbutils.widgets.text("domain_name", "tpch_supply_chain", "domain_name")
dbutils.widgets.text("domain_context", "TPC-H supply chain: customers place orders for parts supplied by suppliers across nations and regions.", "domain_context")
dbutils.widgets.text("source_catalogs", "", "source_catalogs (csv)")
dbutils.widgets.text("source_schemas", "", "source_schemas (csv, catalog.schema)")
dbutils.widgets.text(
    "source_tables",
    ",".join(f"samples.tpch.{t}" for t in ["customer", "orders", "lineitem", "part", "partsupp", "supplier", "nation", "region"]),
    "source_tables (csv, catalog.schema.table)",
)
dbutils.widgets.text("target_catalog", "main", "target_catalog")
dbutils.widgets.text("target_schema", "ontology_tpch", "target_schema")
dbutils.widgets.text("supplemental_volume_dirs", "", "supplemental_volume_dirs (csv)")
dbutils.widgets.text("dashboard_inputs", "", "dashboard_inputs (csv)")
dbutils.widgets.text("source_catalog_for_reads", "", "source_catalog_for_reads")
dbutils.widgets.text("source_schema_for_reads", "", "source_schema_for_reads")
dbutils.widgets.text("warehouse_id", "", "warehouse_id")
dbutils.widgets.text("max_metric_views_per_agent", "50", "max_metric_views_per_agent")
dbutils.widgets.text("repo_path", "", "repo_path")
# Scaffold options
dbutils.widgets.text("target_dir", "", "target_dir (empty = check only)")
dbutils.widgets.text("project_name", "", "project_name (default: target_dir name)")
dbutils.widgets.text("bundle_target", "dev", "bundle_target")
dbutils.widgets.dropdown("force", "false", ["false", "true"], "force overwrite")
dbutils.widgets.text("sdk_root", "", "sdk_root (default: this notebook's repo)")

In [0]:
import json
import sys
from pathlib import Path


def _w(name: str) -> str:
    return dbutils.widgets.get(name).strip()


def _csv(name: str) -> list[str]:
    return [v.strip() for v in _w(name).split(",") if v.strip()]


# SDK root: widget, else two levels above this notebook (<sdk>/notebooks/init_ontology_project).
if _w("sdk_root"):
    sdk_root = Path(_w("sdk_root"))
else:
    nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    sdk_root = Path("/Workspace" + nb_path).parent.parent
if str(sdk_root / "src") not in sys.path:
    sys.path.insert(0, str(sdk_root / "src"))

from genie_ontology_sdk.config import check_access, config_from_dict, validate_config  # noqa: E402
from genie_ontology_sdk.scaffold import scaffold_project  # noqa: E402

raw = {
    "domain_name": _w("domain_name"),
    "domain_context": _w("domain_context"),
    "source_catalogs": _csv("source_catalogs"),
    "source_schemas": _csv("source_schemas"),
    "source_tables": _csv("source_tables"),
    "target_catalog": _w("target_catalog"),
    "target_schema": _w("target_schema"),
    "supplemental_volume_dirs": _csv("supplemental_volume_dirs"),
    "dashboard_inputs": _csv("dashboard_inputs"),
    "source_catalog_for_reads": _w("source_catalog_for_reads"),
    "source_schema_for_reads": _w("source_schema_for_reads"),
    "warehouse_id": _w("warehouse_id"),
    "repo_path": _w("repo_path"),
}
if _w("max_metric_views_per_agent"):
    raw["max_metric_views_per_agent"] = int(_w("max_metric_views_per_agent"))
# Drop empty optional values so the dict matches the schema (no empty strings / lists).
raw = {k: v for k, v in raw.items() if v not in ("", [], None)}
print(f"SDK root: {sdk_root}")
print(json.dumps(raw, indent=2))

In [0]:
from databricks.sdk import WorkspaceClient

config_errors = validate_config(raw)
if config_errors:
    raise ValueError("sdk_config is invalid:\n  " + "\n  ".join(config_errors))
cfg = config_from_dict(raw)

access_problems = check_access(cfg, WorkspaceClient())
print(f"validate_config: {len(config_errors)} errors")
print(f"check_access: {len(access_problems)} problems")
for p in access_problems:
    print(f"  - {p}")
if access_problems:
    raise RuntimeError("Fix the access problems above before scaffolding.")

In [0]:
report = None
if _w("target_dir"):
    report = scaffold_project(
        cfg,
        _w("target_dir"),
        sdk_root,
        force=_w("force") == "true",
        project_name=_w("project_name") or None,
        bundle_target=_w("bundle_target") or "dev",
    )
else:
    print("target_dir is empty: validation and access check only, nothing scaffolded.")

In [0]:
summary = {
    "domain_name": cfg.domain_name,
    "source_inputs": len(cfg.source_catalogs) + len(cfg.source_schemas) + len(cfg.source_tables),
    "config_errors": len(config_errors),
    "access_problems": len(access_problems),
    "scaffold": None if report is None else {
        "target_dir": report.target_dir,
        "files_written": report.files_written,
        "unchanged": len(report.unchanged),
        "skipped": report.skipped,
        "overwritten": report.overwritten,
    },
}
print(json.dumps(summary, indent=2))
if report is not None:
    print("Next: write the gitignored .databricks/bundle/<target>/variable-overrides.json in both bundles (see their READMEs).")
dbutils.notebook.exit(json.dumps(summary))